# Business Entity Resolution (v4), Amazon ML Challenge 2026

v3 scored **0.9853 macro-F0.5 on validation** but **0.939 on the public leaderboard**. v4 is v3 plus fixes for the causes of that gap, found by analysing v3's own test output against the training labels.

## Why v3 lost about 4.6 points on the leaderboard
1. **The data contains deliberate hard negatives.** In train, 29.5% of distractor records (S2/S3 records matching no S1) look almost identical to their best S1: name token-set ≥ 90 and address ≥ 80.
   The typical case is *the same base name plus a descriptor word, at a nearby house number*, i.e. a neighbouring or sister business.
   Examples: `Peak Tri-State Pearl Group, 304B Powell Valley Shores Ln` against S1 `Peak Tri-State Pearl, 299 Powell Valley Shores Lane`, and `1984 Washington Ave` against `1981`.
2. **Test has about 1.9× more distractors per S1 than train**: 5.75 S2/S3 records per S1 against 4.68, with about 3.46 true matches per S1.
   So hard negatives are much more frequent in test than the model and threshold were calibrated for, and each one accepted is a false merge, which F0.5 punishes heavily.
3. **v3 could not tell a neighbouring house number from a noisy one.** Its number features only count exact overlaps, so "299 vs 304" (a different business) looked the same as "5425 vs 1425" (a typo). The labels separate them cleanly:

| look-alike pairs in train | hard negatives | true matches |
|---|---|---|
| every S1 house number present in the record | **4%** | **86%** |
| nearest *different* number is 1–20 away | **57%** | **1%** |
| record adds words to the S1 name | 69% | 16% |

## What v4 changes
| Change | Why / reference |
|---|---|
| **Numeric house-number features** (`hn_*`): common numbers compared *as integers* (so the `003203` vs `3203` zero-padding noise counts as equal), extra and missing numbers, and the distance to the nearest non-shared number, including a "nearby" flag | directly targets the hard negatives above |
| **Target encoding of the extra and missing name words** (`te_*`): for each word the record adds to, or drops from, the S1 name, the smoothed training rate of non-matches, computed out-of-fold | Micci-Barreca, *A preprocessing scheme for high-cardinality categorical attributes*, SIGKDD Explorations 2001. Descriptor words such as *Group*, *Westgate* or *Infratech* signal a sister business |
| **Test-density correction**: distractor pairs are up-weighted by R = (test S2/S3 per S1 − m) / (train S2/S3 per S1 − m), with m = train matches per S1, both in training and in the validation F0.5 used to pick the threshold | prior shift: Saerens, Latinne & Decaestecker, *Adjusting the Outputs of a Classifier to New a Priori Probabilities*, Neural Computation 2002; Elkan, *The Foundations of Cost-Sensitive Learning*, IJCAI 2001 |
| **Stage 2 off by default** (`CFG["stage2"]=False`) | it lost to stage 1 on validation (0.9844 vs 0.9853); turning it off also saves about 2.3 GB of test RAM |
| **Cache reuse**: if a `cache_v3` folder from the v3 run is in `/kaggle/working` or attached under `/kaggle/input`, the normalised sources, lexicon and **test blocking pairs** are reused, because they are unchanged | saves about 50 min |

Everything else is v3: multi-channel blocking (word + char 3-gram TF-IDF), lexicons, segmentation, 3-fold XGBoost, and argmax assignment with the (t, t_low) rescue grid.

**Why no LLM (e.g. Mistral).** *Mistral Small 4* has 119B total parameters (a mixture of experts), which breaks the ≤ 8B rule and does not fit on Kaggle GPUs.
*Ministral 3 8B* is 8.8B including its vision encoder, which is borderline. *Ministral 3 3B* (Apache-2.0) is allowed, but the errors here are synthetic numeric and descriptor-word patterns that a GBDT learns directly from the labels, so a zero-shot 3B model over about 96M pairs adds cost without a clear gain.

**Settings:** GPU T4 ×2, Internet on, Run All. Expect about 2–2.5 h (about 1.5 h if the v3 caches are attached) and a peak of about 22 GB RAM.

In [1]:
!pip install -q rapidfuzz anyascii sparse_dot_topn 2>&1 | tail -2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 12.3 MB/s eta 0:00:00


In [2]:
import gc, glob, json, math, multiprocessing, os, pickle, re, subprocess, sys, time
from collections import Counter, defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
import scipy.sparse as sp
import xgboost as xgb
from anyascii import anyascii
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

multiprocessing.set_start_method("fork", force=True)   # Kaggle default; also makes local runs work
DEBUG = int(os.environ.get("ER_DEBUG", "0"))       # 1 = tiny subsample, for smoke tests only
ON_KAGGLE = os.path.isdir("/kaggle/input")


def find_data_dir():
    if os.environ.get("ER_DATA"):
        return os.environ["ER_DATA"]
    hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
    assert hits, "train_source1.tsv not found under /kaggle/input: add the dataset to the notebook"
    return os.path.dirname(os.path.dirname(hits[0]))   # .../<root>/train/train_source1.tsv -> <root>


DATA = find_data_dir()
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("kaggle_work")
if DEBUG:
    WORK = os.path.join(WORK, "debug")
CACHE, OUT = os.path.join(WORK, "cache_v4"), os.path.join(WORK, "output")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
N_JOBS = os.cpu_count() or 2
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    GPU = False
SEED = 0
CFG = dict(
    k=10,              # word channel: candidates retrieved per S2/S3 record
    max_df=0.02,       # word channel ignores tokens present in >2% of a country's S1
    rel=0.4,           # word channel: drop candidates scoring < 40% of the record's best
    k_char=4,          # v3 char-3gram channel: candidates per record (typos, OCR, glued names)
    max_df_char=0.01,  # char channel ignores 3-grams present in >1% of a country's S1
    rel_char=0.6,      # char channel: drop candidates below 60% of the record's best char score
    frac=0.10,         # training: S1 subset E whose matches are all included  (v2: 0.06)
    other_rate=0.15,   # training: fraction of all other S2/S3 records included (v2: 0.12)
    rounds=1500,
    folds=3,           # out-of-fold stage-1 models
    stage2=False,      # v4: off (lost to stage 1 on validation in v3); True re-enables the v3 stage-2 model
    density="auto",    # v4: distractor up-weighting R for test density ("auto" = from file sizes, or a number; 1.0 = v3)
    te_prior=20.0,     # v4: smoothing strength of the extra/missing-word target encoding
)


def reuse_path(name):
    """A file from an earlier v3 run (unchanged stages): /kaggle/working/cache_v3 or an attached input."""
    hits = [os.path.join(WORK, "cache_v3", name)] + glob.glob(f"/kaggle/input/**/cache_v3/{name}", recursive=True)
    return next((h for h in hits if os.path.exists(h)), None)
T0 = time.time()


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


def ram():
    try:
        with open("/proc/self/status") as f:
            rss = [l for l in f if l.startswith("VmRSS")][0].split()[1]
        return f"{int(rss) / 1e6:.1f} GB"
    except Exception:
        return "?"


log(f"data={DATA} work={WORK} cpus={N_JOBS} gpu={GPU} debug={DEBUG}")

[   0.0 min] data=/kaggle/input/datasets/riceyy/amazon-ml-challenge/student_resource/dataset work=/kaggle/working cpus=4 gpu=True debug=0


## 1. Normalization
This is unchanged from the previous notebook, with two additions.
- **Postal/PIN code** (`pin`), read from the raw address with a country-aware rule. It is the *last* qualifying code, because codes trail addresses and a leading 5-digit US house number must not shadow the real one.
- **Native-script address flag** (bit 8 of `flags`), used by the new address lexicon.

In [3]:
LEGAL = {
    "pvt": "private", "private": "private", "ltd": "limited", "limited": "limited",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "inc": "inc",
    "incorporated": "inc", "corp": "corp", "corporation": "corp", "co": "co",
    "company": "co", "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl",
    "sa": "sa", "sci": "sci", "snc": "snc", "gmbh": "gmbh", "pllc": "pllc",
}
STOP = {"the", "and", "mr", "mrs", "ms", "dr", "sri", "shri", "smt", "m/s", "of",
        "et", "le", "la", "les", "de", "du", "des"}
ALIAS_RE = re.compile(r"\b(?:dba|d/b/a|formerly|aka|f/k/a|fka)\b[:\s]*", re.I)
DOMAIN_RE = re.compile(r"^(?:www\.)?([a-z0-9\-]+)\.(?:com|in|net|org|co|fr|biz|info|us|co\.in)$")
NON_ALNUM = re.compile(r"[^a-z0-9]+")
NUM_RE = re.compile(r"\d+")
NATIVE_RE = re.compile("[ऀ-෿]")        # Devanagari ... Sinhala blocks
ADDR_ABBR = {
    "street": "st", "saint": "st", "road": "rd", "avenue": "ave", "av": "ave", "drive": "dr",
    "circle": "cir", "court": "ct", "boulevard": "blvd", "bd": "blvd", "lane": "ln",
    "place": "pl", "highway": "hwy", "parkway": "pkwy", "terrace": "ter", "trail": "trl",
    "square": "sq", "north": "n", "south": "s", "east": "e", "west": "w",
    "apartment": "apt", "suite": "ste", "floor": "fl", "number": "no", "building": "bldg",
    "chemin": "ch", "impasse": "imp", "allee": "all", "route": "rte",
    "sector": "sec", "opposite": "opp", "near": "nr", "nagar": "ngr",
    "rue": "r",   # v3: French "Rue" and the S2-style "R." abbreviation map to one token
}
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
    "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny",
    "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
    "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
    "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi",
    "wyoming": "wy", "district of columbia": "dc",
}
STATE_RE = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\b")
# v3 postal codes (raw address, commas intact); lookahead so adjacent segments are not consumed
PIN_IN = re.compile(r"\b[1-9]\d{5}\b")                                          # Indian PIN
PIN_US = re.compile(r"(?:\b[A-Z]{2}\s+|(?:^|,)\s*)(\d{5})(?:-\d{4})?\s*(?=,|$)")  # "NC 28655" / ", 28655"
PIN_OTHER = re.compile(r"(?:^|,)\s*(\d{5})(?:\s+[^\d,][^,]*)?\s*(?=,|$)")         # "59000 Lille" / ", 59000"


def extract_pin(raw, country):
    if not raw:
        return ""
    if country == "India":
        m = PIN_IN.findall(raw)
    else:
        m = (PIN_US if country == "US" else PIN_OTHER).findall(raw)
    return m[-1].lstrip("0") if m else ""


def norm_name(raw):
    # returns (core, full, legal, flags); flags: 1 native script, 2 alias, 4 domain
    s = anyascii(raw).lower() if raw else ""
    flags = 1 if NATIVE_RE.search(raw or "") else 0
    m = ALIAS_RE.search(s)
    if m:
        flags |= 2
        s = s[m.end():] or s[: m.start()]
    s = s.strip()
    d = DOMAIN_RE.match(s)
    if d:
        flags |= 4
        s = d.group(1)
    else:
        s = s.replace("&", " and ").replace(".", "")
    toks = NON_ALNUM.sub(" ", s).split()
    core = [t for t in toks if t not in LEGAL and t not in STOP] or toks
    legal = sorted({LEGAL[t] for t in toks if t in LEGAL})
    return " ".join(core), " ".join(toks), " ".join(legal), flags


def norm_addr(raw, country):
    # returns (addr, nums, pin, native_flag)
    s = (anyascii(raw).lower() if raw else "").replace(".", " ")
    s = STATE_RE.sub(lambda m: US_STATES[m.group(1)], s)
    toks = [ADDR_ABBR.get(t, t) for t in NON_ALNUM.sub(" ", s).split()]
    toks = [t for t in toks if t not in ("null", "none", "na")]
    native = 8 if NATIVE_RE.search(raw or "") else 0
    return " ".join(toks), " ".join(sorted(set(NUM_RE.findall(" ".join(toks))))), extract_pin(raw, country), native


NORM_COLS = ["core", "full", "legal", "flags", "addr", "nums", "pin"]


def _norm_chunk(args):
    names, addrs, ctrys = args
    rows = []
    for n, a, c in zip(names, addrs, ctrys):
        core, full, legal, flags = norm_name(n)
        addr, nums, pin, native = norm_addr(a, c)
        rows.append((core, full, legal, flags | native, addr, nums, pin))
    df = pd.DataFrame(rows, columns=NORM_COLS)
    df["flags"] = df["flags"].astype(np.int8)
    return df.astype({c: "string[pyarrow]" for c in NORM_COLS if c != "flags"})


def read_tsv(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=3)


def load_source(split, src):
    # normalized source file with ids + country only (raw text dropped to save RAM); cached
    cp = os.path.join(CACHE, f"{split}_s{src}.parquet")
    if not os.path.exists(cp) and reuse_path(f"{split}_s{src}.parquet"):
        cp = reuse_path(f"{split}_s{src}.parquet")              # v3 normalisation is unchanged
    if os.path.exists(cp):
        return pd.read_parquet(cp, dtype_backend="pyarrow")
    df = read_tsv(os.path.join(DATA, split, f"{split}_source{src}.tsv"))
    names, addrs, ctrys = df.business_name.tolist(), df.business_address.tolist(), df.country.tolist()
    step = 100_000
    chunks = [(names[i:i + step], addrs[i:i + step], ctrys[i:i + step]) for i in range(0, len(df), step)]
    del names, addrs, ctrys
    with Pool(N_JOBS) as pool:
        norm = pd.concat(pool.imap(_norm_chunk, chunks), ignore_index=True)
    out = pd.concat([df[["entity_id", "country"]].astype("string[pyarrow]").reset_index(drop=True), norm], axis=1)
    del df, chunks, norm; gc.collect()
    out.to_parquet(cp, index=False)
    return out


def load_split(split):
    s1 = load_source(split, 1)
    q = pd.concat([load_source(split, 2), load_source(split, 3)], ignore_index=True)
    q["is_s3"] = q.entity_id.str.startswith("S3-").astype(np.int8)
    return s1, q


def load_truth():
    # Series: S2/S3 id -> its S1 id (each S2/S3 matches at most one S1)
    gt = read_tsv(os.path.join(DATA, "train", "train_ground_truth.tsv"))
    s, m = [], []
    for a, b in zip(gt.source1_entity_id, gt.matched_entity_ids):
        for x in (b.split(",") if b else []):
            s.append(a); m.append(x)
    return pd.Series(s, index=m)


for sp_ in ("train", "test"):
    for src in (1, 2, 3):
        n = len(load_source(sp_, src)); gc.collect()
        log(f"normalized {sp_} source{src}: {n} rows, RAM {ram()}")

[   0.0 min] normalized train source1: 2206821 rows, RAM 0.8 GB
[   0.1 min] normalized train source2: 5034616 rows, RAM 1.5 GB
[   0.1 min] normalized train source3: 5285603 rows, RAM 1.7 GB
[   0.1 min] normalized test source1: 1732544 rows, RAM 1.1 GB
[   0.2 min] normalized test source2: 4887273 rows, RAM 1.9 GB
[   0.2 min] normalized test source3: 5082316 rows, RAM 1.8 GB


## 2. Lexicons mined from training matches, and glued-word segmentation
- **Names:** the previous co-occurrence lexicon, plus **positional alignment**. When a romanised native-script name and its matched S1 name have the same token count, token *i* is aligned to token *i* (Sajjad et al. 2012).
  This gives the mapping for each token, not just its most frequent co-occurring word, so `praivet`→`private` does not collapse to `limited`.
- **Addresses (new):** a romanised native-script address token (such as a state name) that is absent from the S1 address vocabulary is mapped to the S1 tokens it co-occurs with in ≥80% of matched pairs.
- **Segmentation (new):** a glued S2/S3 name token (≥8 letters, not an S1 word) is split into S1-vocabulary words by unigram DP (Norvig 2009). Example: `darkbuildwellclinic` → `dark buildwell clinic`.

In [4]:
def mine_lexicon(s1, q, truth, min_count=3, min_share=0.5):
    tgt = q["entity_id"].astype(str).map(truth)
    has = tgt.notna().to_numpy()
    flags = q["flags"].astype(int).to_numpy()
    s1_full = pd.Series(s1["full"].astype(str).to_numpy(), index=s1["entity_id"].astype(str))
    s1_addr = pd.Series(s1["addr"].astype(str).to_numpy(), index=s1["entity_id"].astype(str))

    # ---- names: co-occurrence (previous notebook) + positional alignment (v3) ----
    m = ((flags & 1) == 1) & has
    src_names = q["full"].astype(str).to_numpy()[m]
    tgt_names = s1_full.reindex(tgt[m]).fillna("").to_numpy()
    co, cnt, pco, pcnt = defaultdict(Counter), Counter(), defaultdict(Counter), Counter()
    for a, b in zip(src_names, tgt_names):
        at, bt = a.split(), b.split()
        bs = set(bt)
        for t in set(at):
            cnt[t] += 1
            co[t].update(bs)
        if len(at) == len(bt):
            for x, y_ in zip(at, bt):
                pcnt[x] += 1
                pco[x][y_] += 1
    lex = {}
    for t, c in cnt.items():
        if c >= min_count:
            best, n = co[t].most_common(1)[0]
            if n / c >= min_share and best != t:
                lex[t] = best
    n_pos = 0
    for t, c in pcnt.items():
        if c >= 2:
            best, n = pco[t].most_common(1)[0]
            if n / c >= min_share and best != t:
                lex[t] = best            # positional alignment wins over co-occurrence
                n_pos += 1

    # ---- addresses: romanised native tokens -> co-occurring S1 address tokens (v3) ----
    s1_vocab = set()
    for x in s1["addr"].astype(str):
        s1_vocab.update(x.split())
    m = ((flags & 8) == 8) & has
    src_addr = q["addr"].astype(str).to_numpy()[m]
    tgt_addr = s1_addr.reindex(tgt[m]).fillna("").to_numpy()
    aco, acnt = defaultdict(Counter), Counter()
    for a, b in zip(src_addr, tgt_addr):
        bs = set(b.split())
        for t in set(a.split()):
            if t not in s1_vocab and not t.isdigit():
                acnt[t] += 1
                aco[t].update(bs)
    alex = {}
    for t, c in acnt.items():
        if c >= 5:
            top = [w for w, n in aco[t].most_common(3) if n / c >= 0.8]
            if top:
                alex[t] = " ".join(top)
    log(f"lexicon: {len(lex)} name tokens ({n_pos} positional), {len(alex)} address tokens")
    return {"name": lex, "addr": alex}


def _core_legal(toks):
    return (" ".join([t for t in toks if t not in LEGAL and t not in STOP] or toks),
            " ".join(sorted({LEGAL[t] for t in toks if t in LEGAL})))


def apply_lexicon(df, lex):
    flags = df["flags"].astype(int).to_numpy()
    mask = (flags & 1) == 1
    cores, fulls, legals = [], [], []
    for f in df.loc[mask, "full"].astype(str).tolist():
        toks = " ".join(lex["name"].get(t, t) for t in f.split()).split()
        c, l = _core_legal(toks)
        cores.append(c); fulls.append(" ".join(toks)); legals.append(l)
    for col, vals in (("core", cores), ("full", fulls), ("legal", legals)):
        df.loc[mask, col] = pd.array(vals, dtype=df[col].dtype)
    amask = (flags & 8) == 8
    if amask.any() and lex["addr"]:
        vals = [" ".join(lex["addr"].get(t, t) for t in a.split()) for a in df.loc[amask, "addr"].astype(str).tolist()]
        df.loc[amask, "addr"] = pd.array(vals, dtype=df["addr"].dtype)
    return df


_SEG_VOCAB = {}


def _seg_words(words):
    out = []
    for w in words:
        n = len(w)
        best = [None] * (n + 1)
        best[0] = (0.0, [])
        for i in range(n):
            if best[i] is None:
                continue
            for j in range(i + 2, min(n, i + 20) + 1):
                f = _SEG_VOCAB.get(w[i:j])
                if f:
                    sc = best[i][0] + math.log(f) - 8.0
                    if best[j] is None or sc > best[j][0]:
                        best[j] = (sc, best[i][1] + [w[i:j]])
        seg = best[n][1] if best[n] else None
        out.append(" ".join(seg) if seg and 2 <= len(seg) <= 5 else None)
    return out


def segment_glued(s1, q):
    """Split glued S2/S3 name tokens into S1-vocabulary words (vocabulary from this split's S1)."""
    global _SEG_VOCAB
    cnt = Counter()
    for x in s1["core"].astype(str):
        cnt.update(x.split())
    _SEG_VOCAB = {t: c for t, c in cnt.items() if c >= 3 and len(t) >= 2}
    cands = set()
    for x in q["core"].astype(str):
        for t in x.split():
            if len(t) >= 8 and t.isalpha() and t not in _SEG_VOCAB:
                cands.add(t)
    cands = sorted(cands)
    if not cands:
        return q
    parts = [cands[i::N_JOBS] for i in range(N_JOBS)]
    with Pool(N_JOBS) as pool:                # fork: workers inherit _SEG_VOCAB
        outs = pool.map(_seg_words, parts)
    seg = {w: s for p, o in zip(parts, outs) for w, s in zip(p, o) if s}
    for col in ("core", "full"):
        vals = q[col].astype(str).tolist()
        hit = [i for i, x in enumerate(vals) if any(t in seg for t in x.split())]
        if hit:
            new = [" ".join(seg.get(t, t) for t in vals[i].split()) for i in hit]
            q.loc[q.index[hit], col] = pd.array(new, dtype=q[col].dtype)
        del vals
    log(f"segmentation: {len(seg)} of {len(cands)} glued tokens split")
    return q


def prepare(split):
    s1, q = load_split(split)
    apply_lexicon(q, LEX)
    segment_glued(s1, q)
    gc.collect()
    return s1, q


LEX_PATH = os.path.join(CACHE, "lexicon.pkl")
if not os.path.exists(LEX_PATH) and reuse_path("lexicon.pkl"):
    LEX_PATH = reuse_path("lexicon.pkl")
if not os.path.exists(LEX_PATH):
    _s1, _q = load_split("train")
    LEX = mine_lexicon(_s1, _q, load_truth())
    pickle.dump(LEX, open(LEX_PATH, "wb"))
    del _s1, _q; gc.collect()
LEX = pickle.load(open(LEX_PATH, "rb"))
log(f"lexicon {len(LEX['name'])} name / {len(LEX['addr'])} address tokens, e.g. "
    + ", ".join(f"{k}->{v}" for k, v in list(LEX["name"].items())[:6])
    + " | " + ", ".join(f"{k}->{v}" for k, v in list(LEX["addr"].items())[:4]))

[   0.2 min] lexicon 572 name / 413 address tokens, e.g. modrn->modern, phud->food, riyl->real, limitet->limited, kulopl->global, picins->business | mharastr->maharashtra, mdhy->pradesh madhya, prdes->pradesh, pscimbng->w bengal


## 3. Multi-channel blocking: each S2/S3 record retrieves S1 candidates within its country
- **Word channel** (previous notebook): TF-IDF over tagged name and address tokens, top-10 per record, with relative pruning.
- **Char channel (new):** TF-IDF over character 3-grams of the space-less core name, top-4 per record, keeping only candidates within 60% of the best char score.
  Character n-grams are the standard typo-tolerant blocking key (Christen 2012). They catch OCR swaps, typos, glued words and romanisation variants, where the word channel shares no token.

The two channels are **unioned**, and every surviving pair gets **exact cosines from four views**: word (`score`), char (`cos_c`), name-only (`cos_n`) and address-only (`cos_a`).
All four are features. Identical documents are transformed once, and every country is processed separately to bound RAM.

In [5]:
def _docs(df):
    name = df["core"].fillna("").astype(str).str.replace(r"(\S+)", r"n_\1", regex=True)
    addr = df["addr"].fillna("").astype(str).str.replace(r"(\S+)", r"a_\1", regex=True)
    return name + " " + addr


def _squash(df):
    return " " + df["core"].fillna("").astype(str).str.replace(" ", "", regex=False) + " "


def _fit(vec, docs):
    try:
        return vec.fit_transform(docs).tocsr()
    except ValueError:        # empty vocabulary (tiny debug subsets)
        return None


def _transform(vec, uniq, chunk=500_000):
    return sp.vstack([vec.transform(list(uniq[a:a + chunk])) for a in range(0, len(uniq), chunk)]).tocsr()


def _retrieve(Q, ST, k, rel, thr, chunk=400_000):
    # top-k per row of Q with relative pruning; returns (row, col) of kept pairs
    rows, cols, vals = [], [], []
    for a in range(0, Q.shape[0], chunk):
        C = sp_matmul_topn(Q[a:a + chunk], ST, top_n=k, sort=True, threshold=thr, n_threads=N_JOBS).tocoo()
        rows.append(C.row.astype(np.int64) + a); cols.append(C.col.astype(np.int64)); vals.append(C.data.astype(np.float32))
    rows, cols, vals = np.concatenate(rows), np.concatenate(cols), np.concatenate(vals)
    best = np.zeros(Q.shape[0], np.float32)
    np.maximum.at(best, rows, vals)
    keep = vals >= rel * best[rows]
    return rows[keep], cols[keep]


def _expand(codes, rows, cols, n_uniq):
    # unique-doc results -> every query having that doc: (query_local, s1_local)
    if len(rows) == 0:
        return np.empty(0, np.int64), np.empty(0, np.int64)
    o = np.argsort(rows, kind="stable"); rows, cols = rows[o], cols[o]
    starts = np.searchsorted(rows, np.arange(n_uniq))
    sizes = np.diff(np.r_[starts, len(rows)])
    qs = sizes[codes]
    pos = np.repeat(starts[codes], qs) + (np.arange(qs.sum()) - np.repeat(np.cumsum(qs) - qs, qs))
    return np.repeat(np.arange(len(codes), dtype=np.int64), qs), cols[pos]


def _member(key_sorted, sub):
    if len(sub) == 0:
        return np.zeros(len(key_sorted), np.int8)
    s = np.sort(sub)
    i = np.minimum(np.searchsorted(s, key_sorted), len(s) - 1)
    return (s[i] == key_sorted).astype(np.int8)


def _rowdot(A, B, ia, ib, chunk=1_000_000):
    out = np.empty(len(ia), np.float32)
    for a in range(0, len(ia), chunk):
        out[a:a + chunk] = np.asarray(A[ia[a:a + chunk]].multiply(B[ib[a:a + chunk]]).sum(1)).ravel()
    return out


def _rank_in_group(g, v):
    # rank (0 = best) of v within groups g (g sorted ascending)
    o = np.lexsort((-v, g))
    gs = g[o]
    starts = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    sizes = np.diff(np.r_[starts, len(gs)])
    r = np.empty(len(g), np.int64)
    r[o] = np.arange(len(g)) - np.repeat(starts, sizes)
    return r


def _cos_view(s_text, q_text, ql, sl):
    vec = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True, dtype=np.float32)
    S = _fit(vec, s_text.fillna("").astype(str).tolist())
    if S is None:
        return np.zeros(len(ql), np.float32)
    codes, uniq = pd.factorize(q_text.fillna("").astype(str))
    Q = _transform(vec, uniq)
    out = _rowdot(Q, S, codes[ql], sl)
    del Q, S, vec; gc.collect()
    return out


def block(s1, q, k=CFG["k"], max_df=CFG["max_df"], rel=CFG["rel"]):
    out, rec = [], {"word": 0, "char": 0, "union": 0}
    q_ctry, s_ctry = q["country"].astype(str).to_numpy(), s1["country"].astype(str).to_numpy()
    for country in pd.unique(q_ctry):                    # open set of countries, taken from the data
        s_idx, q_idx = np.flatnonzero(s_ctry == country), np.flatnonzero(q_ctry == country)
        if len(s_idx) == 0 or len(q_idx) == 0:
            continue
        n_s = len(s_idx)
        s_sub, q_sub = s1.iloc[s_idx], q.iloc[q_idx]
        # word channel
        vw = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True, max_df=max_df, dtype=np.float32)
        Sw = vw.fit_transform(_docs(s_sub).tolist()).tocsr()
        cw, uw = pd.factorize(_docs(q_sub))
        log(f"  block {country}: {n_s} S1, {len(q_idx)} queries ({len(uw)} unique docs)")
        Qw = _transform(vw, uw)
        rw, colw = _retrieve(Qw, Sw.T.tocsr(), k, rel, 0.05)
        qlw, slw = _expand(cw, rw, colw, len(uw)); del rw, colw
        # char channel
        vc = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False, sublinear_tf=True,
                             max_df=CFG["max_df_char"], dtype=np.float32)
        Sc = _fit(vc, _squash(s_sub).tolist())
        cc_, uc = pd.factorize(_squash(q_sub))
        if Sc is not None:
            Qc = _transform(vc, uc)
            rc, colc = _retrieve(Qc, Sc.T.tocsr(), CFG["k_char"], CFG["rel_char"], 0.1)
            qlc, slc = _expand(cc_, rc, colc, len(uc)); del rc, colc
        else:
            Qc, qlc, slc = None, np.empty(0, np.int64), np.empty(0, np.int64)
        kw, kc = qlw * n_s + slw, qlc * n_s + slc
        del qlw, slw, qlc, slc
        key = np.union1d(kw, kc)                          # sorted by (query, s1)
        in_w, in_c = _member(key, kw), _member(key, kc)
        del kw, kc
        ql, sl = key // n_s, key % n_s
        del key
        score = _rowdot(Qw, Sw, cw[ql], sl)
        cos_c = _rowdot(Qc, Sc, cc_[ql], sl) if Sc is not None else np.zeros(len(ql), np.float32)
        del Qw, Sw, Qc, Sc, vw, vc; gc.collect()
        cos_n = _cos_view(s_sub["core"], q_sub["core"], ql, sl)
        cos_a = _cos_view(s_sub["addr"], q_sub["addr"], ql, sl)
        rank, c_rank = _rank_in_group(ql, score), _rank_in_group(ql, cos_c)
        out.append(pd.DataFrame({"qi": q_idx[ql].astype(np.int32), "si": s_idx[sl].astype(np.int32),
                                 "score": score, "rank": np.minimum(rank, 127).astype(np.int8),
                                 "cos_c": cos_c, "c_rank": np.minimum(c_rank, 127).astype(np.int8),
                                 "cos_n": cos_n, "cos_a": cos_a, "in_w": in_w, "in_c": in_c}))
        del ql, sl, score, cos_c, cos_n, cos_a, rank, c_rank, in_w, in_c, s_sub, q_sub; gc.collect()
        log(f"  block {country}: done, {len(out[-1])} pairs "
            f"(word {int(out[-1].in_w.sum())}, char-only {int(((out[-1].in_c == 1) & (out[-1].in_w == 0)).sum())}), RAM {ram()}")
    P = pd.concat(out, ignore_index=True)
    del out; gc.collect()
    P = P.sort_values(["qi", "rank"], kind="stable").reset_index(drop=True)
    return add_context(P)


def add_context(P):
    # per-query context: best score, gap to runner-up, number of candidates, gaps on the other views
    qi, sc = P["qi"].to_numpy(), P["score"].to_numpy()
    starts = np.r_[0, np.flatnonzero(np.diff(qi)) + 1]
    sizes = np.diff(np.r_[starts, len(qi)])
    best = np.repeat(sc[starts], sizes)
    second = np.repeat(np.where(sizes > 1, sc[np.minimum(starts + 1, len(sc) - 1)], 0.0), sizes)
    P["q_best_ratio"] = (sc / np.maximum(best, 1e-6)).astype(np.float32)
    P["q_gap_best"] = (best - sc).astype(np.float32)
    P["q_gap_next"] = np.where(P["rank"].to_numpy() == 0, sc - second, sc - best).astype(np.float32)
    P["q_ncand"] = np.repeat(sizes, sizes).astype(np.int16)
    for c, name in (("cos_c", "c_gap_best"), ("cos_n", "n_gap_best"), ("cos_a", "a_gap_best")):
        v = P[c].to_numpy()
        P[name] = (np.repeat(np.maximum.reduceat(v, starts), sizes) - v).astype(np.float32)
    return P

## 4. Pair features
This is the v3 feature set, with two new groups (see the top of the notebook).
- **`hn_*`, house numbers as integers:** up to 6 numbers per address.
  - `hn_common`: numbers shared, compared as integers, so zero-padding counts as equal.
  - `hn_q_extra`, `hn_s_missing`: numbers only in the record, or only in the S1.
  - `hn_s_all_in_q`: every S1 number appears in the record.
  - `hn_min_diff`: log of the distance between the closest *non-shared* numbers.
  - `hn_nearby`: that distance is between 1 and 20, the neighbouring-business signature.
- **`te_*`, target encoding of name-word differences:** for words the record adds (`te_extra_*`) or drops (`te_missing_max`) relative to the S1 core name, the smoothed training rate of non-matches.
  The rates are computed out-of-fold on training rows, from all training rows for validation and test, and stored by word so every country's vocabulary can use them.

Country is still **not** a feature.

In [6]:
CTX = ["score", "rank", "q_best_ratio", "q_gap_best", "q_gap_next", "q_ncand",
       "cos_c", "c_rank", "cos_n", "cos_a", "c_gap_best", "n_gap_best", "a_gap_best", "in_w", "in_c"]
KEEP_X = ["n_tsort", "n_squash_ratio", "a_tset", "num_jac", "pin_match", "pin_conflict"]   # reused by stage 2
TE_COLS = ["te_extra_max", "te_extra_mean", "te_missing_max"]
HN_W = 6                                  # house numbers kept per address


def _num_array(nums):
    # "12 305 0042" -> int32 row [12, 305, 42, -1, -1, -1]; last 9 digits keep values inside int32
    out = np.full((len(nums), HN_W), -1, np.int32)
    for i, x in enumerate(nums):
        if x:
            v = [int(t[-9:]) for t in x.split()[:HN_W]]
            out[i, :len(v)] = v
    return out


class Featurizer:
    # country is deliberately not a feature, so the model transfers to unseen countries (France)
    def __init__(self, s1, q):
        cols = ("core", "full", "legal", "addr", "nums", "pin")
        self.s = {c: s1[c].fillna("").astype(str).to_numpy(dtype=object) for c in cols}
        self.q = {c: q[c].fillna("").astype(str).to_numpy(dtype=object) for c in cols}
        self.q_flags = q["flags"].astype(np.int8).to_numpy()
        self.q_s3 = q["is_s3"].astype(np.int8).to_numpy()
        key = s1["country"].astype(str) + "|" + s1["core"].astype(str)
        self.s_name_freq = key.map(key.value_counts()).astype(np.int32).to_numpy()
        self.mats = {}
        for c in ("core", "addr", "nums"):     # binary token matrices -> vectorized overlap counts
            cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, dtype=np.float32)
            S = cv.fit_transform(self.s[c]).tocsr()
            Q = cv.transform(self.q[c]).tocsr()
            ls = np.array([x.count(" ") + 1 if x else 0 for x in self.s[c]], dtype=np.float32)
            lq = np.array([x.count(" ") + 1 if x else 0 for x in self.q[c]], dtype=np.float32)
            self.mats[c] = (S, Q, ls, lq)
            if c == "core":
                self.core_vocab = cv.get_feature_names_out()
        self.qn, self.sn = _num_array(self.q["nums"]), _num_array(self.s["nums"])
        self.te_extra = self.te_missing = None      # set by set_te(); zeros until then

    def set_te(self, te):
        # te: {"extra": {word: rate}, "missing": {word: rate}, "prior": p0} -> vectors over this vocabulary
        p0 = te["prior"]
        self.te_extra = np.array([te["extra"].get(t, p0) for t in self.core_vocab], dtype=np.float32)
        self.te_missing = np.array([te["missing"].get(t, p0) for t in self.core_vocab], dtype=np.float32)

    def diff_mats(self, qi, si):
        # binary sparse matrices of words the record adds (extra) / drops (missing) vs. the S1 core name
        S, Q, _, _ = self.mats["core"]
        Qm, Sm = Q[qi], S[si]
        inter = Qm.multiply(Sm)
        extra, missing = (Qm - inter).tocsr(), (Sm - inter).tocsr()
        extra.eliminate_zeros(); missing.eliminate_zeros()
        return extra, missing

    def te_feats(self, qi, si, te_extra, te_missing):
        extra, missing = self.diff_mats(qi, si)
        n_ex = np.asarray(extra.sum(1)).ravel()
        ex_w = sp.csr_matrix(extra.multiply(te_extra[None, :]))    # csr_matrix(): robust to scipy returning coo or dense
        mi_w = sp.csr_matrix(missing.multiply(te_missing[None, :]))
        ex_max = np.asarray(ex_w.max(1).todense()).ravel()
        ex_mean = np.where(n_ex > 0, np.asarray(ex_w.sum(1)).ravel() / np.maximum(n_ex, 1), -1)
        mi_max = np.where(np.asarray(missing.sum(1)).ravel() > 0, np.asarray(mi_w.max(1).todense()).ravel(), -1)
        return (np.where(n_ex > 0, ex_max, -1).astype(np.float32), ex_mean.astype(np.float32), mi_max.astype(np.float32))

    def hn_feats(self, qi, si):
        A, B = self.qn[qi], self.sn[si]
        va, vb = A >= 0, B >= 0
        eq = (A[:, :, None] == B[:, None, :]) & va[:, :, None] & vb[:, None, :]
        a_in_b, b_in_a = eq.any(2), eq.any(1)
        na, nb = va.sum(1), vb.sum(1)
        common = (a_in_b & va).sum(1)
        s_missing = nb - (b_in_a & vb).sum(1)
        um = va[:, :, None] & vb[:, None, :] & ~a_in_b[:, :, None] & ~b_in_a[:, None, :]
        big = np.iinfo(np.int64).max
        d = np.where(um, np.abs(A[:, :, None].astype(np.int64) - B[:, None, :].astype(np.int64)), big).min((1, 2))
        has = d < big
        return {"hn_common": common.astype(np.int8), "hn_q_extra": (na - common).astype(np.int8),
                "hn_s_missing": s_missing.astype(np.int8),
                "hn_s_all_in_q": np.where(nb > 0, (s_missing == 0).astype(np.int8), -1).astype(np.int8),
                "hn_min_diff": np.where(has, np.log1p(np.where(has, d, 0)), -1).astype(np.float32),
                "hn_nearby": (has & (d <= 20)).astype(np.int8)}

    def _overlap(self, c, qi, si):
        S, Q, ls, lq = self.mats[c]
        inter = np.asarray(Q[qi].multiply(S[si]).sum(1)).ravel()
        union = lq[qi] + ls[si] - inter
        return inter, np.where(union > 0, inter / np.maximum(union, 1), 0), lq[qi], ls[si]

    def transform(self, P):
        qi, si = P["qi"].to_numpy(), P["si"].to_numpy()
        F = {c: P[c].to_numpy() for c in CTX}
        kw = dict(workers=N_JOBS, dtype=np.float32)
        qc, sc = self.q["core"][qi], self.s["core"][si]
        F["n_ratio"] = cpdist(qc, sc, scorer=fuzz.ratio, **kw)
        F["n_tsort"] = cpdist(qc, sc, scorer=fuzz.token_sort_ratio, **kw)
        F["n_tset"] = cpdist(qc, sc, scorer=fuzz.token_set_ratio, **kw)
        F["n_partial"] = cpdist(qc, sc, scorer=fuzz.partial_ratio, **kw)
        F["n_jw"] = cpdist(qc, sc, scorer=JaroWinkler.normalized_similarity, **kw)
        qsq = np.array([x.replace(" ", "") for x in qc], dtype=object)
        ssq = np.array([x.replace(" ", "") for x in sc], dtype=object)
        F["n_squash_ratio"] = cpdist(qsq, ssq, scorer=fuzz.ratio, **kw)
        F["n_squash_partial"] = cpdist(qsq, ssq, scorer=fuzz.partial_ratio, **kw)
        F["n_full_tsort"] = cpdist(self.q["full"][qi], self.s["full"][si], scorer=fuzz.token_sort_ratio, **kw)
        inter, jac, lq, ls = self._overlap("core", qi, si)
        F.update(n_inter=inter, n_jac=jac, n_lq=lq, n_ls=ls, n_missing_s=ls - inter, n_missing_q=lq - inter)
        ql, sl = self.q["legal"][qi], self.s["legal"][si]
        both = (ql != "") & (sl != "")
        F["legal_both"] = both.astype(np.int8)
        F["legal_eq"] = (both & (ql == sl)).astype(np.int8)
        F["legal_sim"] = cpdist(ql, sl, scorer=fuzz.token_set_ratio, **kw) * both
        qa, sa = self.q["addr"][qi], self.s["addr"][si]
        F["a_ratio"] = cpdist(qa, sa, scorer=fuzz.ratio, **kw)
        F["a_tset"] = cpdist(qa, sa, scorer=fuzz.token_set_ratio, **kw)
        F["a_tsort"] = cpdist(qa, sa, scorer=fuzz.token_sort_ratio, **kw)
        F["a_partial"] = cpdist(qa, sa, scorer=fuzz.partial_ratio, **kw)
        inter, jac, lq, ls = self._overlap("addr", qi, si)
        F.update(a_inter=inter, a_jac=jac, a_lq=lq, a_ls=ls)
        inter, jac, lq, ls = self._overlap("nums", qi, si)
        F.update(num_inter=inter, num_jac=jac, num_lq=lq, num_ls=ls)
        F["num_tset"] = cpdist(self.q["nums"][qi], self.s["nums"][si], scorer=fuzz.token_set_ratio, **kw)
        qp, spn = self.q["pin"][qi], self.s["pin"][si]
        pboth = (qp != "") & (spn != "")
        F["pin_both"] = pboth.astype(np.int8)
        F["pin_match"] = (pboth & (qp == spn)).astype(np.int8)
        F["pin_conflict"] = (pboth & (qp != spn)).astype(np.int8)
        F["pin_prefix3"] = (pboth & (np.array([a[:3] for a in qp], dtype=object) == np.array([b[:3] for b in spn], dtype=object))).astype(np.int8)
        flags = self.q_flags[qi]
        F["q_native"], F["q_alias"], F["q_domain"] = flags & 1, (flags >> 1) & 1, (flags >> 2) & 1
        F["q_addr_native"] = (flags >> 3) & 1
        F["q_is_s3"] = self.q_s3[qi]
        F["s_name_freq"] = self.s_name_freq[si]
        F.update(self.hn_feats(qi, si))
        if self.te_extra is not None:
            F["te_extra_max"], F["te_extra_mean"], F["te_missing_max"] = self.te_feats(qi, si, self.te_extra, self.te_missing)
        else:                                   # training: filled out-of-fold after featurisation
            F["te_extra_max"] = F["te_extra_mean"] = F["te_missing_max"] = np.zeros(len(qi), np.float32)
        # competition: this candidate vs. the query's best alternative (chunks never split a query)
        for c in ("n_tsort", "n_squash_ratio", "a_tset", "num_jac", "hn_min_diff"):
            v = pd.Series(np.asarray(F[c], dtype=np.float32))
            F[c + "_gap_qmax"] = (v.groupby(qi).transform("max").to_numpy() - v.to_numpy()).astype(np.float32)
        return pd.DataFrame(F)


def chunk_bounds(qi, size):
    # ~size-row chunks that never split one query's candidates
    cuts = [0]
    while cuts[-1] < len(qi):
        c = min(cuts[-1] + size, len(qi))
        while 0 < c < len(qi) and qi[c] == qi[c - 1]:
            c += 1
        cuts.append(c)
    return list(zip(cuts[:-1], cuts[1:]))


def featurize(feat, P, size=1_000_000):
    X, names = None, None
    for a, b in chunk_bounds(P["qi"].to_numpy(), size):
        F = feat.transform(P.iloc[a:b])
        if X is None:
            names = list(F.columns)
            X = np.empty((len(P), len(names)), dtype=np.float32)
        X[a:b] = F.to_numpy(dtype=np.float32)
        del F
    return X, names

## 5. Stage 2 (collective features) and decoding
**Stage 2** (stacking, Wolpert 1992; link features, Saeedi et al. 2018) takes the stage-1 probability of every candidate pair and adds:
- the record's own competition: its rank, its gap to the next-best S1, and how many candidates it has with p>0.5;
- the **S1 side**: how many confident records the S1 already has, the best competing record for that S1, and this record's rank within the S1;
- the strength of the competing S1.

All of these come from arrays of the stage-1 probabilities (a sort plus bincounts), so no string features are recomputed.

**Decoding:** each S2/S3 record goes only to its argmax S1, and only if p > t. v3 adds a **rescue** step: an S1 left with no match takes its best argmax-assigned record if p > t_low.
About 94% of S1 are non-singletons, so an empty prediction is usually wrong (Lipton et al. 2014; Dembczyński et al. 2011). Both t and t_low are tuned for macro F0.5.

In [7]:
def _top2(g, v, n_groups, other=None):
    # per-row rank of v within group g, per-group max / second max (0 if absent), and `other` at ranks 0 / 1
    o = np.lexsort((-v, g))
    gs, vs = g[o], v[o]
    starts = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1] if len(gs) else np.empty(0, np.int64)
    sizes = np.diff(np.r_[starts, len(gs)])
    r_sorted = np.arange(len(gs)) - np.repeat(starts, sizes)
    rank = np.empty(len(g), np.int32); rank[o] = r_sorted
    m1 = np.zeros(n_groups, np.float32); m2 = np.zeros(n_groups, np.float32)
    m1[gs[starts]] = vs[starts]
    sec = r_sorted == 1
    m2[gs[sec]] = vs[sec]
    o0 = o1 = None
    if other is not None:
        os_ = other[o]
        o0 = np.full(n_groups, -1, np.int64); o1 = np.full(n_groups, -1, np.int64)
        o0[gs[starts]] = os_[starts]; o1[gs[sec]] = os_[sec]
    return rank, m1, m2, o0, o1


S2_CTX = ["score", "rank", "q_gap_next", "q_ncand", "cos_c", "cos_n", "cos_a"]


class Stage2:
    """Collective features from stage-1 probabilities p (aligned with P rows)."""

    def __init__(self, qi, si, p, n_s1):
        qi, si = qi.astype(np.int64), si.astype(np.int64)
        n_q = int(qi.max()) + 1 if len(qi) else 1
        self.qi, self.si, self.p = qi, si, p.astype(np.float32)
        self.rkq, self.m1q, self.m2q, self.top_s0, self.top_s1 = _top2(qi, self.p, n_q, other=si)
        self.rks, self.m1s, self.m2s, _, _ = _top2(si, self.p, n_s1)
        hi = self.p > 0.5
        self.tophi = ((self.rkq == 0) & hi).astype(np.float32)
        self.q_hi = np.bincount(qi, weights=hi, minlength=n_q).astype(np.float32)
        self.s_hi = np.bincount(si, weights=hi, minlength=n_s1).astype(np.float32)
        self.s_sum = np.bincount(si, weights=self.p, minlength=n_s1).astype(np.float32)
        self.s_top = np.bincount(si, weights=self.tophi, minlength=n_s1).astype(np.float32)
        self.names = (["p1", "p1_rank_q", "p1_gap_q", "q_n_hi", "s_n_hi", "s_sum_p", "s_rank_p", "s_max_other",
                       "s_n_top", "s_n_top_other", "comp_n_top", "comp_n_hi"] + S2_CTX + KEEP_X)

    def rows(self, a, b, P, keepx):
        q, s, p = self.qi[a:b], self.si[a:b], self.p[a:b]
        rkq, rks = self.rkq[a:b], self.rks[a:b]
        comp = np.where(rkq == 0, self.top_s1[q], self.top_s0[q])
        cols = [p, rkq, p - np.where(rkq == 0, self.m2q[q], self.m1q[q]), self.q_hi[q],
                self.s_hi[s], self.s_sum[s], rks, np.where(rks == 0, self.m2s[s], self.m1s[s]),
                self.s_top[s], self.s_top[s] - self.tophi[a:b],
                np.where(comp >= 0, self.s_top[np.maximum(comp, 0)], -1), np.where(comp >= 0, self.s_hi[np.maximum(comp, 0)], -1)]
        cols += [P[c].to_numpy()[a:b] for c in S2_CTX] + [keepx[a:b, j] for j in range(keepx.shape[1])]
        return np.column_stack([np.asarray(c, dtype=np.float32) for c in cols])


def argmax_per_query(qi, prob):
    order = np.lexsort((-prob, qi))
    first = np.r_[True, qi[order][1:] != qi[order][:-1]]
    return order[first]


def select(best, prob, si, t, t_low, n_s1):
    # indices (into P rows) of the final matches: argmax rows with p > t, plus rescue of empty S1s
    bp, bs = prob[best], si[best]
    sel = best[bp > t]
    if t_low < t:
        covered = np.zeros(n_s1, bool); covered[si[sel]] = True
        c = np.flatnonzero((bp > t_low) & (bp <= t) & ~covered[bs])
        if len(c):
            c = c[np.lexsort((-bp[c], bs[c]))]
            c = c[np.r_[True, bs[c][1:] != bs[c][:-1]]]
            sel = np.concatenate([sel, best[c]])
    return sel


def predict_rows(bst, X, rows, chunk=2_000_000):
    out = np.empty(len(rows), np.float32)
    for a in range(0, len(rows), chunk):
        out[a:a + chunk] = bst.inplace_predict(X[rows[a:a + chunk]], iteration_range=(0, bst.best_iteration + 1))
    return out

## 6. Training at full S1 density
The sampling protocol is unchanged: a query sample is blocked against **all** training S1, E is a random S1 subset whose matches are all included, V ⊂ E is held out, and sampled false positives are up-weighted to population scale.
The sample is larger than before (frac 0.10, other_rate 0.15).

Stage 1 is **3 XGBoost models on query-level folds**, giving out-of-fold probabilities for every training row. Validation rows and the test set use the average of the 3 models.
Stage 2 is trained on the out-of-fold probabilities. Both stages are tuned on V with the (t, t_low) grid, and the better one is kept.

In [8]:
REPORT_PATH = os.path.join(CACHE, "train_report.json")
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                  device="cuda" if GPU else "cpu", max_depth=10, learning_rate=0.08, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, max_bin=256, seed=SEED,
                  nthread=N_JOBS)
XGB2_PARAMS = dict(XGB_PARAMS, max_depth=6, learning_rate=0.05)
T_GRID = np.round(np.arange(0.05, 0.961, 0.025), 3)
TLOW_GRID = [1.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3]     # 1.0 = no rescue (previous behaviour)

def density_ratio(truth):
    """R = test distractors per S1 / train distractors per S1, assuming test has train's matches per S1."""
    import pyarrow.parquet as pq
    n = {(sp_, s): pq.ParquetFile(load_path(sp_, s)).metadata.num_rows for sp_ in ("train", "test") for s in (1, 2, 3)}
    m = len(truth) / n[("train", 1)]
    tr = (n[("train", 2)] + n[("train", 3)]) / n[("train", 1)]
    te = (n[("test", 2)] + n[("test", 3)]) / n[("test", 1)]
    r = float(np.clip((te - m) / max(tr - m, 1e-6), 1.0, 4.0))
    log(f"density: train {tr:.2f} / test {te:.2f} S2+S3 records per S1, {m:.2f} matches per S1 -> distractor weight R={r:.2f}")
    return r


def load_path(split, src):
    p = os.path.join(CACHE, f"{split}_s{src}.parquet")
    return p if os.path.exists(p) else reuse_path(f"{split}_s{src}.parquet")


def fill_te_oof(feat, P, X, FEATURES, y, n_tr, fold, K, prior_a, chunk=1_000_000):
    """Out-of-fold target encoding of extra/missing name words, estimated on each record's top candidate."""
    V = len(feat.core_vocab)
    cnt_e, neg_e = np.zeros((K, V)), np.zeros((K, V))
    cnt_m, neg_m = np.zeros((K, V)), np.zeros((K, V))
    pqi, psi, rk = P.qi.to_numpy(), P.si.to_numpy(), P["rank"].to_numpy()
    top_tr = np.flatnonzero((rk[:n_tr] == 0))
    for a in range(0, len(top_tr), chunk):
        r = top_tr[a:a + chunk]
        extra, missing = feat.diff_mats(pqi[r], psi[r])
        neg = (y[r] == 0).astype(np.float64)
        for k in range(K):
            mk = (fold[r] == k).astype(np.float64)
            cnt_e[k] += extra.T @ mk; neg_e[k] += extra.T @ (mk * neg)
            cnt_m[k] += missing.T @ mk; neg_m[k] += missing.T @ (mk * neg)
    p0 = float((y[top_tr] == 0).mean()) if len(top_tr) else 0.5
    enc = lambda n_, c_: ((n_ + prior_a * p0) / (c_ + prior_a)).astype(np.float32)
    te_all = (enc(neg_e.sum(0), cnt_e.sum(0)), enc(neg_m.sum(0), cnt_m.sum(0)))
    te_fold = [(enc(neg_e.sum(0) - neg_e[k], cnt_e.sum(0) - cnt_e[k]), enc(neg_m.sum(0) - neg_m[k], cnt_m.sum(0) - cnt_m[k]))
               for k in range(K)]
    cols = [FEATURES.index(c) for c in TE_COLS]
    row_fold = np.full(len(P), -1); row_fold[:n_tr] = fold
    for a, b in chunk_bounds(pqi, chunk):
        for k in list(range(K)) + [-1]:
            r = np.arange(a, b)[row_fold[a:b] == k]
            if len(r):
                te_e, te_m = te_fold[k] if k >= 0 else te_all
                X[r[:, None], cols] = np.column_stack(feat.te_feats(pqi[r], psi[r], te_e, te_m))
    vocab = feat.core_vocab
    return {"extra": dict(zip(vocab, te_all[0].tolist())), "missing": dict(zip(vocab, te_all[1].tolist())), "prior": p0}


if not os.path.exists(REPORT_PATH):
    rng = np.random.default_rng(SEED)
    s1, q = prepare("train")
    truth = load_truth()
    R = density_ratio(truth) if CFG["density"] == "auto" else float(CFG["density"])
    frac, other_rate = CFG["frac"], CFG["other_rate"]
    if DEBUG:                               # smoke test: tiny S1 world
        keep_s1 = rng.random(len(s1)) < 0.05
        ids = set(s1.entity_id.astype(str)[keep_s1])
        tq = q.entity_id.astype(str).map(truth)
        q = q[(tq.isin(ids) | (tq.isna() & (rng.random(len(q)) < 0.05))).to_numpy()].reset_index(drop=True)
        s1 = s1[keep_s1].reset_index(drop=True)
        frac, other_rate = 0.3, 0.5
    s1id = s1.entity_id.astype(str).to_numpy()
    in_e = rng.random(len(s1)) < frac
    in_v = in_e & (rng.random(len(s1)) < 0.25)
    pos = pd.Series(np.arange(len(s1)), index=s1id)
    t_pos = q.entity_id.astype(str).map(truth).map(pos).fillna(-1).astype(np.int64).to_numpy()
    q_e = (t_pos >= 0) & in_e[np.maximum(t_pos, 0)]
    keep = q_e | (~q_e & (rng.random(len(q)) < other_rate))
    q, t_pos, q_e = q[keep].reset_index(drop=True), t_pos[keep], q_e[keep]
    q_val = np.where(q_e, in_v[np.maximum(t_pos, 0)], rng.random(len(q)) < 0.25)
    log(f"train sample: {len(q)} queries ({q_e.sum()} matched to E), E={in_e.sum()} V={in_v.sum()} S1")

    P = block(s1, q)
    pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
    y = (t_pos[pqi] == psi).astype(np.float32)
    has = t_pos >= 0
    found = np.zeros(len(q), bool); found[pqi[y == 1]] = True
    found_w = np.zeros(len(q), bool); found_w[pqi[(y == 1) & (P.in_w.to_numpy() == 1)]] = True
    log(f"pairs {len(P)} ({len(P) / len(q):.2f}/query), blocking recall {found[has].mean():.4f} "
        f"(word channel alone {found_w[has].mean():.4f})")

    order = np.argsort(q_val[pqi], kind="stable")          # train rows first, val rows last
    P, y = P.iloc[order].reset_index(drop=True), y[order]
    pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
    n_tr = int((~q_val[pqi]).sum())
    feat = Featurizer(s1, q)
    X, FEATURES = featurize(feat, P)
    K = CFG["folds"]
    fold = rng.integers(0, K, size=len(q))[pqi[:n_tr]]
    TE = fill_te_oof(feat, P, X, FEATURES, y, n_tr, fold, K, CFG["te_prior"])
    pickle.dump(TE, open(os.path.join(CACHE, "target_encoding.pkl"), "wb"))
    del feat; gc.collect()
    # sampling weights; true distractors (no S1 at all) are also up-weighted to the TEST density (R)
    is_dis = t_pos < 0
    w = np.where(q_e[pqi], 1.0 / frac, np.where(is_dis[pqi], R, 1.0) / other_rate).astype(np.float32)
    w /= w.mean()
    log(f"features {X.shape}, train rows {n_tr}, R={R:.2f}, RAM {ram()}")

    # ---------------- stage 1: K query-level folds -> out-of-fold probabilities ----------------
    va_rows = np.arange(n_tr, len(P))
    p1 = np.zeros(len(P), np.float32)
    best_its = []
    for k in range(K):
        tr_rows = np.flatnonzero(fold != k)
        dtr = xgb.QuantileDMatrix(X[tr_rows], y[tr_rows], weight=w[tr_rows], feature_names=FEATURES)
        dva = xgb.QuantileDMatrix(X[n_tr:], y[n_tr:], weight=w[n_tr:], ref=dtr, feature_names=FEATURES)
        bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=CFG["rounds"] if not DEBUG else 60,
                        evals=[(dva, "val")], early_stopping_rounds=50, verbose_eval=200)
        del dtr, dva; gc.collect()
        ho = np.flatnonzero(fold == k)
        p1[ho] = predict_rows(bst, X, ho)
        p1[n_tr:] += predict_rows(bst, X, va_rows) / K
        best_its.append(int(bst.best_iteration))
        bst.save_model(os.path.join(CACHE, f"xgb_fold{k}.json"))
        if k == 0:
            imp = pd.Series(bst.get_score(importance_type="gain")).sort_values(ascending=False)
            print((imp / imp.sum()).head(15).round(4).to_string())
        del bst; gc.collect()
        log(f"stage-1 fold {k}: best iteration {best_its[-1]}, RAM {ram()}")
    keepx = X[:, [FEATURES.index(c) for c in KEEP_X]].copy() if CFG["stage2"] else None
    del X; gc.collect()

    # ---------------- validation metric (population-weighted, as before) ----------------
    vqi, vsi = pqi[n_tr:], psi[n_tr:]
    v_idx = np.flatnonzero(in_v)
    n_true = np.bincount(t_pos[has & q_e & q_val], minlength=len(s1))[v_idx]
    # sampled distractors -> population scale, and true distractors -> TEST density (x R)
    w_fp = np.where(q_e, 1.0, np.where(t_pos < 0, R, 1.0) / (other_rate * 0.25))

    def evaluator(prob):
        best = argmax_per_query(vqi, prob)

        def f(t, t_low):
            sel = select(best, prob, vsi, t, t_low, len(s1))
            s_sel, q_sel = vsi[sel], vqi[sel]
            m = in_v[s_sel]; s_sel, q_sel = s_sel[m], q_sel[m]
            ok = (t_pos[q_sel] == s_sel).astype(float)
            tp = np.bincount(s_sel, weights=ok, minlength=len(s1))[v_idx]
            npred = np.bincount(s_sel, weights=ok + (1 - ok) * w_fp[q_sel], minlength=len(s1))[v_idx]
            return np.where(n_true == 0, (npred == 0).astype(float),
                            1.25 * tp / np.maximum(npred + 0.25 * n_true, 1e-9))
        return f

    def tune(prob, name):
        f = evaluator(prob)
        res = {(t, tl): f(t, tl).mean() for t in T_GRID for tl in TLOW_GRID if tl < t or tl == 1.0}
        (bt, btl), bf = max(res.items(), key=lambda kv: kv[1])
        no_rescue = max(v for (t, tl), v in res.items() if tl == 1.0)
        log(f"{name}: best macro-F0.5 {bf:.5f} at t={bt}, t_low={btl} (without rescue {no_rescue:.5f})")
        return float(bt), float(btl), float(bf), f

    t1, tl1, f1, ev1 = tune(p1[n_tr:], "stage 1")
    use2, t2, tl2, f2 = False, t1, tl1, f1
    S2_FEATURES, best_it2 = None, None
    if CFG["stage2"]:
        st = Stage2(pqi, psi, p1, len(s1))
        X2 = st.rows(0, len(P), P, keepx)
        S2_FEATURES = st.names
        del st; gc.collect()
        d2tr = xgb.QuantileDMatrix(X2[:n_tr], y[:n_tr], weight=w[:n_tr], feature_names=S2_FEATURES)
        d2va = xgb.QuantileDMatrix(X2[n_tr:], y[n_tr:], weight=w[n_tr:], ref=d2tr, feature_names=S2_FEATURES)
        bst2 = xgb.train(XGB2_PARAMS, d2tr, num_boost_round=1000 if not DEBUG else 60,
                         evals=[(d2va, "val")], early_stopping_rounds=50, verbose_eval=200)
        p2 = predict_rows(bst2, X2, va_rows)
        best_it2 = int(bst2.best_iteration)
        bst2.save_model(os.path.join(CACHE, "xgb_stage2.json"))
        imp2 = pd.Series(bst2.get_score(importance_type="gain")).sort_values(ascending=False)
        print((imp2 / imp2.sum()).head(12).round(4).to_string())
        del d2tr, d2va, X2, bst2; gc.collect()
        t2, tl2, f2, ev2 = tune(p2, "stage 2")
        use2 = f2 > f1
    t_best, tl_best = (t2, tl2) if use2 else (t1, tl1)
    fs = (ev2 if use2 else ev1)(t_best, tl_best)
    ctry = s1.country.astype(str).to_numpy()[v_idx]
    report = {"use_stage2": bool(use2), "threshold": t_best, "t_low": tl_best,
              "val_macro_f05": float(fs.mean()), "val_macro_f05_stage1": f1, "val_macro_f05_stage2": f2 if CFG["stage2"] else None,
              "val_f05_singletons": float(fs[n_true == 0].mean()) if (n_true == 0).any() else None,
              "val_f05_non_singletons": float(fs[n_true > 0].mean()),
              "val_f05_by_country": {c: float(fs[ctry == c].mean()) for c in np.unique(ctry)},
              "blocking_recall": float(found[has].mean()), "blocking_recall_word_only": float(found_w[has].mean()),
              "pairs_per_query": float(len(P) / len(q)), "best_iterations": best_its, "best_iteration_stage2": best_it2,
              "density_R": R,
              "features": FEATURES, "stage2_features": S2_FEATURES, "folds": K}
    json.dump(report, open(REPORT_PATH, "w"), indent=1)
    del s1, q, P, p1, keepx, TE; gc.collect()

REPORT = json.load(open(REPORT_PATH))
print(json.dumps({k: v for k, v in REPORT.items() if k not in ("features", "stage2_features")}, indent=1))
TE = pickle.load(open(os.path.join(CACHE, "target_encoding.pkl"), "rb"))

[   1.5 min] segmentation: 718091 of 774896 glued tokens split
[   1.8 min] density: train 4.68 / test 5.75 S2+S3 records per S1, 3.46 matches per S1 -> distractor weight R=1.89
[   2.1 min] train sample: 2197416 queries (762738 matched to E), E=220479 V=55247 S1
[   2.6 min]   block India: 883188 S1, 879001 queries (852029 unique docs)
[   8.6 min]   block India: done, 8482342 pairs (word 5685750, char-only 2796592), RAM 4.5 GB
[   9.0 min]   block US: 1323633 S1, 1318415 queries (1245960 unique docs)
[  23.4 min]   block US: done, 11510227 pairs (word 8028983, char-only 3481244), RAM 5.0 GB
[  23.4 min] pairs 19992569 (9.10/query), blocking recall 0.9810 (word channel alone 0.9741)
[  29.7 min] features (19992569, 69), train rows 14982902, R=1.89, RAM 10.9 GB
[0]	val-logloss:0.20266
[200]	val-logloss:0.00608
[400]	val-logloss:0.00578
[600]	val-logloss:0.00574
[738]	val-logloss:0.00573


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [14:36:31] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


q_gap_next                 0.3929
rank                       0.1793
hn_nearby                  0.1005
hn_min_diff                0.0454
q_gap_best                 0.0411
q_best_ratio               0.0371
a_tset_gap_qmax            0.0344
te_extra_max               0.0319
n_squash_ratio_gap_qmax    0.0124
hn_common                  0.0112
hn_q_extra                 0.0071
c_rank                     0.0059
te_extra_mean              0.0058
num_tset                   0.0056
hn_s_all_in_q              0.0053
[  32.8 min] stage-1 fold 0: best iteration 688, RAM 11.3 GB
[0]	val-logloss:0.20272
[200]	val-logloss:0.00619
[400]	val-logloss:0.00594
[600]	val-logloss:0.00588
[746]	val-logloss:0.00588
[  36.1 min] stage-1 fold 1: best iteration 696, RAM 11.3 GB
[0]	val-logloss:0.20273
[200]	val-logloss:0.00619
[400]	val-logloss:0.00593
[600]	val-logloss:0.00591
[672]	val-logloss:0.00590
[  39.1 min] stage-1 fold 2: best iteration 622, RAM 11.3 GB
[  39.3 min] stage 1: best macro-F0.5 0.98820 at t=

## 7. Test inference → `output/matching_results.tsv` + `output/candidate_pairs.tsv`
Countries are scored one at a time. Stage-1 probability is the mean of the 3 fold models, and only the 6 `KEEP_X` feature columns are kept for stage 2.
`candidate_pairs.tsv` holds every pair the model scored, so the matches are always a subset of it, rescued ones included.

In [9]:
def write_lists(path, col, s1_ids, q_ids, si, qi, block_size=200_000):
    # one row per S1 (input order), comma-joined S2/S3 ids; strings built per block to save RAM
    o = np.lexsort((qi, si)); si, qi = si[o], qi[o]
    keep = np.r_[True, (si[1:] != si[:-1]) | (qi[1:] != qi[:-1])]
    si, qi = si[keep], qi[keep]
    bounds = np.searchsorted(si, np.arange(len(s1_ids) + 1))
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for a in range(0, len(s1_ids), block_size):
            b = min(a + block_size, len(s1_ids))
            ids = q_ids[qi[bounds[a]:bounds[b]]]
            loc = bounds[a:b + 1] - bounds[a]
            f.write("".join(f"{s1_ids[i]}\t{','.join(ids[loc[i - a]:loc[i - a + 1]])}\n" for i in range(a, b)))


s1, q = prepare("test")
if DEBUG:
    rng = np.random.default_rng(1)
    s1 = s1[rng.random(len(s1)) < 0.05].reset_index(drop=True)
    q = q[rng.random(len(q)) < 0.05].reset_index(drop=True)
PAIRS_PATH = os.path.join(CACHE, "test_pairs.parquet")
if not os.path.exists(PAIRS_PATH) and reuse_path("test_pairs.parquet"):
    PAIRS_PATH = reuse_path("test_pairs.parquet")                # v3 blocking is unchanged
    log(f"reusing v3 test blocking pairs: {PAIRS_PATH}")
if not os.path.exists(PAIRS_PATH):
    block(s1, q).to_parquet(PAIRS_PATH, index=False)
    gc.collect()
P = pd.read_parquet(PAIRS_PATH)
log(f"test pairs {len(P)} ({len(P) / len(q):.2f}/query), RAM {ram()}")

FEATURES = REPORT["features"]
P1_PATH, KX_PATH, PROB_PATH = (os.path.join(CACHE, f) for f in ("test_p1.npy", "test_keepx.npy", "test_prob.npy"))
if not os.path.exists(P1_PATH):
    boosters = []
    for k in range(REPORT["folds"]):
        b = xgb.Booster(); b.load_model(os.path.join(CACHE, f"xgb_fold{k}.json"))
        b.set_param({"device": "cuda" if GPU else "cpu", "nthread": N_JOBS})
        boosters.append((b, REPORT["best_iterations"][k]))
    p1 = np.zeros(len(P), dtype=np.float32)
    keepx = np.zeros((len(P), len(KEEP_X)), dtype=np.float32) if REPORT["use_stage2"] else None
    kx_idx = [FEATURES.index(c) for c in KEEP_X]
    # v4 memory: integer country codes (v3 built a 96M-element Python-string array) and 1M-row slices
    # of P per chunk (v3 copied a whole country's pairs, ~2.8 GB for India, before scoring)
    q_code, ctry_names = pd.factorize(q.country.astype(str))
    s_code = pd.Index(ctry_names).get_indexer(s1.country.astype(str))
    pair_code = q_code.astype(np.int16)[P.qi.to_numpy()]
    for ci, c in enumerate(ctry_names):                 # one country at a time keeps RAM low
        rows = np.flatnonzero(pair_code == ci)
        if len(rows) == 0:
            continue
        s_idx, q_idx = np.flatnonzero(s_code == ci), np.flatnonzero(q_code == ci)
        s_loc = np.full(len(s1), -1, np.int64); s_loc[s_idx] = np.arange(len(s_idx))
        q_loc = np.full(len(q), -1, np.int64); q_loc[q_idx] = np.arange(len(q_idx))
        feat = Featurizer(s1.iloc[s_idx].reset_index(drop=True), q.iloc[q_idx].reset_index(drop=True))
        feat.set_te(TE)
        # P is sorted by qi, so a country's rows are too: chunks never split a query
        for a, b in chunk_bounds(P.qi.to_numpy()[rows], 1_000_000):
            Pc = P.iloc[rows[a:b]].copy()
            Pc["qi"], Pc["si"] = q_loc[Pc.qi.to_numpy()], s_loc[Pc.si.to_numpy()]
            X = feat.transform(Pc)[FEATURES].to_numpy(dtype=np.float32)
            p1[rows[a:b]] = np.mean([bb.inplace_predict(X, iteration_range=(0, it + 1)) for bb, it in boosters], axis=0)
            if keepx is not None:
                keepx[rows[a:b]] = X[:, kx_idx]
            del X, Pc
        log(f"  scored {c}: {len(rows)} pairs, RAM {ram()}")
        del feat, rows; gc.collect()
    np.save(P1_PATH, p1)
    if keepx is not None:
        np.save(KX_PATH, keepx)
    del boosters, keepx; gc.collect()
p1 = np.load(P1_PATH)
keepx = np.load(KX_PATH) if REPORT["use_stage2"] else None

pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
if REPORT["use_stage2"]:
    if not os.path.exists(PROB_PATH):
        bst2 = xgb.Booster(); bst2.load_model(os.path.join(CACHE, "xgb_stage2.json"))
        bst2.set_param({"device": "cuda" if GPU else "cpu", "nthread": N_JOBS})
        st = Stage2(pqi, psi, p1, len(s1))
        prob = np.empty(len(P), np.float32)
        for a in range(0, len(P), 2_000_000):
            b = min(a + 2_000_000, len(P))
            prob[a:b] = bst2.inplace_predict(st.rows(a, b, P, keepx), iteration_range=(0, REPORT["best_iteration_stage2"] + 1))
        np.save(PROB_PATH, prob)
        del st, bst2; gc.collect()
    prob = np.load(PROB_PATH)
else:
    prob = p1
log(f"scoring done ({'stage 2' if REPORT['use_stage2'] else 'stage 1'}), RAM {ram()}")

THRESHOLD, T_LOW = REPORT["threshold"], REPORT["t_low"]   # change here to trade precision vs recall, then re-run this cell
sel = select(argmax_per_query(pqi, prob), prob, psi, THRESHOLD, T_LOW, len(s1))
s1id = s1.entity_id.astype(str).to_numpy()
qid = q.entity_id.astype(str).to_numpy()
write_lists(os.path.join(OUT, "matching_results.tsv"), "matched_entity_ids", s1id, qid, psi[sel], pqi[sel])
write_lists(os.path.join(OUT, "candidate_pairs.tsv"), "candidate_entity_ids", s1id, qid, psi, pqi)
n = np.bincount(psi[sel], minlength=len(s1))
ctry = s1.country.astype(str).to_numpy()
for c in np.unique(ctry):
    m = ctry == c
    log(f"  {c}: {m.sum()} S1, mean matches {n[m].mean():.2f}, empty rate {(n[m] == 0).mean():.3f}")
log(f"threshold {THRESHOLD}, t_low {T_LOW}: wrote {OUT}/matching_results.tsv and candidate_pairs.tsv")

[  40.6 min] segmentation: 604077 of 637656 glued tokens split
[  40.6 min] reusing v3 test blocking pairs: /kaggle/working/cache_v3/test_pairs.parquet
[  40.8 min] test pairs 95706481 (9.60/query), RAM 16.9 GB
[  58.7 min]   scored India: 46270148 pairs, RAM 18.2 GB
[  64.4 min]   scored France: 18003016 pairs, RAM 14.6 GB
[  74.8 min]   scored US: 31433317 pairs, RAM 16.3 GB
[  74.8 min] scoring done (stage 1), RAM 14.5 GB
[  76.9 min]   France: 259452 S1, mean matches 3.74, empty rate 0.031
[  76.9 min]   India: 809986 S1, mean matches 3.42, empty rate 0.053
[  76.9 min]   US: 663106 S1, mean matches 3.53, empty rate 0.047
[  76.9 min] threshold 0.225, t_low 0.1: wrote /kaggle/working/output/matching_results.tsv and candidate_pairs.tsv


## 8. Validate the submission files

In [10]:
m = pd.read_csv(os.path.join(OUT, "matching_results.tsv"), sep="\t", dtype=str, keep_default_na=False)
assert list(m.columns) == ["source1_entity_id", "matched_entity_ids"]
assert m.source1_entity_id.is_unique and set(m.source1_entity_id) == set(s1id), "S1 coverage mismatch"
lists = m.matched_entity_ids[m.matched_entity_ids != ""].str.split(",")
assert lists.map(lambda l: len(l) == len(set(l))).all(), "duplicate ids in a list"
assert lists.explode().str.match(r"^S[23]-").all(), "non S2/S3 id"
print(f"basic checks OK: {len(m)} rows, {(m.matched_entity_ids == '').sum()} empty")

val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True) + glob.glob("utils/validate_submission.py")
if val and not DEBUG:
    subprocess.run([sys.executable, val[0], "--matching", os.path.join(OUT, "matching_results.tsv"),
                    "--candidate", os.path.join(OUT, "candidate_pairs.tsv"),
                    "--test-dir", os.path.join(DATA, "test")], check=False)
else:
    print("official validator not found (add utils/validate_submission.py to the dataset to run it)")
print("Download /kaggle/working/output/matching_results.tsv and upload it to the portal.")

basic checks OK: 1732544 rows, 81698 empty
ML Challenge 2026 — submission validator
  test dir: /kaggle/input/datasets/riceyy/amazon-ml-challenge/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (81698 empty, 1650846 non-empty).
  candidate_pairs.tsv: 1732544 rows (99 empty, 1732445 non-empty).

PASS — no blocking issues found. Safe to submit.
Download /kaggle/working/output/matching_results.tsv and upload it to the portal.
